The following code is written by Zhizuo Chen (aka George Chen).

It pre-processes the data file "train_sent_emo_dya.csv":

https://github.com/declare-lab/MELD/blob/master/data/MELD_Dyadic/train_sent_emo_dya.csv

In [1]:
import numpy as np, pandas as pd, time, math, random, scipy
from collections import OrderedDict
from sklearn.preprocessing import LabelEncoder

In [2]:
#Function to change timestamps in "HH:MM:SS,mmm" to miliseconds.
def timestamp2int(time_stamp_str):

    #Get the hour, minute, second, and millisecond.
    hour, minute, rest = time_stamp_str.split(':')
    second, millisecond = rest.split(',')
    #Get the second in float format.
    return 1000 * (3600 * int(hour) + 60 * int(minute) + int(second)) + int(millisecond)
    #func-end

In [3]:
#Read the data.
df = pd.read_csv("train_sent_emo_dya.csv", header = 0)
#Transform the 'StartTime' and 'EndTime' to float numbers.
df['StartTime_int'] = df['StartTime'].apply(timestamp2int)
df['EndTime_int'] = df['EndTime'].apply(timestamp2int)
#Show the data.
df.head()

,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Old_Dialogue_ID,Old_Utterance_ID,Season,Episode,StartTime,EndTime,StartTime_int,EndTime_int
0,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,0,0,8,21,"00:16:16,059","00:16:21,731",976059,981731
1,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,0,1,8,21,"00:16:21,940","00:16:23,442",981940,983442
2,That I did. That I did.,Chandler,neutral,neutral,0,2,0,2,8,21,"00:16:23,442","00:16:26,389",983442,986389
3,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,0,3,8,21,"00:16:26,820","00:16:29,572",986820,989572
4,My duties? All right.,Chandler,surprise,positive,0,4,0,4,8,21,"00:16:34,452","00:16:40,917",994452,1000917


In [4]:
#Drop irrelavant columns.
df.drop(columns = ['Sentiment', 'Dialogue_ID', 'Utterance_ID', 'Old_Dialogue_ID', 'Old_Utterance_ID', 'StartTime', 'EndTime'], inplace = True)
#Move the 'Utterance' column to the last. 
col = df.pop('Utterance')
df.insert(len(df.columns), 'Utterance', col)
#Drop duplicates.
df.drop_duplicates(inplace = True)
#Show the data.
df.head()

,Speaker,Emotion,Season,Episode,StartTime_int,EndTime_int,Utterance
0,Chandler,neutral,8,21,976059,981731,also I was the point person on my company’s tr...
1,The Interviewer,neutral,8,21,981940,983442,You must’ve had your hands full.
2,Chandler,neutral,8,21,983442,986389,That I did. That I did.
3,The Interviewer,neutral,8,21,986820,989572,So let’s talk a little bit about your duties.
4,Chandler,surprise,8,21,994452,1000917,My duties? All right.


In [5]:
#Check the speakers.
dfv = df.values
#The new list.
n_dfv = list()
#The count of all names.
nam_ct_d = OrderedDict()

#Loop through all rows.
for row in dfv:
    #Get the speaker name.
    spk_nam = row[0]
    #In the case of 'Joey/Drake', drop the name after '/'.
    spk_nam = spk_nam.split('/')[0]
    #Check whether the name string contains ' and '.
    names = spk_nam.split(' and ')
    #In the case of 'Joey and Ross', duplicate the records.
    for name in names:
        v = nam_ct_d.get(name)
        if v is None: nam_ct_d[name] = 1
        else: nam_ct_d[name] = v + 1
        n_dfv.append(np.array([name, *row[1: ]], dtype = object))
        #inner-for-end
    #for-end
#Reform the dataframe.
n_df = pd.DataFrame(n_dfv, columns = df.columns)
#Drop duplicates again (safer).
n_df.drop_duplicates(inplace = True)

In [6]:
#Order the names descendantly in their frequencies.
nam_order = np.flip(np.argsort(list(nam_ct_d.values())))
nam_desc = np.array(list(nam_ct_d.keys()), dtype = object)[nam_order]
#Encode the names.
nam_cod_d = OrderedDict()
for i, name in enumerate(nam_desc): nam_cod_d[name] = i
#Insert the encoded names.
n_df.insert(n_df.columns.get_loc('Episode') + 1, 'Speaker_coded', n_df['Speaker'].apply(lambda name: nam_cod_d[name]))
#Encode the emotions.
cod_emotion = LabelEncoder()
n_df.insert(n_df.columns.get_loc('Speaker_coded') + 1, 'Emotion_coded', cod_emotion.fit_transform(n_df['Emotion'].values))
#Drop the original 'Speaker' and 'Emotion' columns.
n_df.drop(columns = ['Speaker', 'Emotion'], inplace = True)
#Sorting the data.
n_df.sort_values(by = ['Season', 'Episode', 'StartTime_int'], ignore_index = True, inplace = True)
#Show the data.
n_df.head()

,Season,Episode,Speaker_coded,Emotion_coded,StartTime_int,EndTime_int,Utterance
0,1,1,5,4,79037,85417,"Alright, so I'm back in high school, I'm stand..."
1,1,1,11,4,85627,87169,"Oh, yeah. Had that dream."
2,1,1,5,6,87378,91465,"Then I look down, and I realize there's a phon..."
3,1,1,0,6,94928,95600,Instead of...?
4,1,1,5,4,95600,96892,That's right.


In [7]:
#Save the data file in csv format.
n_df.to_csv("emo_data.csv", header = True, index = False)
#Save the segmented data files by 'Season' and 'Episode' in csv format.
for (s, e), sub_df in n_df.groupby(['Season', 'Episode']):
    sub_df.to_csv("emo_data_s%02d_e%02d.csv" % (s, e), header = True, index = False)
    #for-end

In [8]:
#Display the encoding of 'Speaker'.
print("Speaker\tCode")
for k, v in nam_cod_d.items(): print(k, '\t', v)

Speaker	Code
Joey 	 0
Rachel 	 1
Ross 	 2
Phoebe 	 3
Monica 	 4
Chandler 	 5
Janice 	 6
Carol 	 7
Emily 	 8
Tag 	 9
Mona 	 10
All 	 11
Pete 	 12
Mark 	 13
Danny 	 14
Richard 	 15
Doug 	 16
Joanna 	 17
Mr. Treeger 	 18
Mr. Geller 	 19
Julie 	 20
Elizabeth 	 21
Mrs. Geller 	 22
Dr. Green 	 23
Phoebe Sr 	 24
Susan 	 25
Paul 	 26
Eric 	 27
Frank 	 28
Woman 	 29
Barry 	 30
Chip 	 31
Gunther 	 32
Kate 	 33
David 	 34
Joshua 	 35
Cassie 	 36
Charlie 	 37
Kristen 	 38
Mr. Tribbiani 	 39
Ben 	 40
Russell 	 41
Dina 	 42
The Casting Director 	 43
Katie 	 44
Julio 	 45
Policeman 	 46
Man 	 47
Jill 	 48
Lydia 	 49
Issac 	 50
Stanley 	 51
Kathy 	 52
The Interviewer 	 53
Receptionist 	 54
Guy 	 55
The Cooking Teacher 	 56
Janine 	 57
Mischa 	 58
Robert 	 59
Earl 	 60
Dr. Baldhara 	 61
Mike 	 62
Isabella 	 63
Mr. Heckles 	 64
Cliff 	 65
Chloe 	 66
Dana 	 67
Krista 	 68
Nurse 	 69
Duncan 	 70
Gary 	 71
Mr. Franklin 	 72
Kim 	 73
Shelley 	 74
Ursula 	 75
Megan 	 76
The Director 	 77
Jim 	 78
Larry 	 79


In [9]:
#Display the encoding of 'Emotion'.
print("Emotion\tCode")
for i, s in enumerate(cod_emotion.classes_): print(s, '\t', i)

Emotion	Code
anger 	 0
disgust 	 1
fear 	 2
joy 	 3
neutral 	 4
sadness 	 5
surprise 	 6
